# Controlador Proporcional Derivativo

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import ipywidgets as widgets
from IPython.display import display, clear_output

# -------------------------------------------------------
# Parámetros
# -------------------------------------------------------

kp = 1.4                       # Ganancia proporcional fija
t = np.linspace(0, 15, 800)    # Vector de tiempo

# Ruido de medición
noise_amplitude = 0.02
noise_frequency = 15           # rad/s


# -------------------------------------------------------
# Controles interactivos
# -------------------------------------------------------

kd_slider = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=8.0,
    step=0.1,
    description='kd:',
    continuous_update=False,
    readout_format='.1f',
    layout=widgets.Layout(width='500px')
)

noise_checkbox = widgets.Checkbox(
    value=False,
    description='Activar ruido de medición'
)

output = widgets.Output()


# -------------------------------------------------------
# Tiempo de establecimiento al 2 %
# -------------------------------------------------------

def settling_time(t, y, y_final):

    band = 0.02 * abs(y_final)

    outside = np.where(
        np.abs(y - y_final) > band
    )[0]

    if len(outside) == 0:
        return 0.0

    if outside[-1] >= len(t) - 1:
        return np.nan

    return t[outside[-1] + 1]


# -------------------------------------------------------
# Simulación
# -------------------------------------------------------

def simulate(change=None):

    kd = kd_slider.value
    noise_on = noise_checkbox.value

    # ---------------------------------------------------
    # Polos
    #
    # s² + (1.4 + kd)s + (1 + kp) = 0
    # ---------------------------------------------------

    poles = np.roots(
        [1, 1.4 + kd, 1 + kp]
    )


    # ---------------------------------------------------
    # Respuesta SIN ruido
    #
    # La derivada se aplica sobre la medición para evitar
    # el "derivative kick" producido por una referencia
    # escalón.
    #
    #              kp
    # T(s) = -------------------------
    #        s²+(1.4+kd)s+(1+kp)
    # ---------------------------------------------------

    system_clean = signal.TransferFunction(
        [kp],
        [1, 1.4 + kd, 1 + kp]
    )

    tout, y_clean = signal.step(
        system_clean,
        T=t
    )


    # ---------------------------------------------------
    # Ruido
    # ---------------------------------------------------

    if noise_on:

        n = noise_amplitude * np.sin(noise_frequency * t)

        dn = (
            noise_amplitude
            * noise_frequency
            * np.cos(noise_frequency * t)
        )

    else:

        n = np.zeros_like(t)
        dn = np.zeros_like(t)


    # ---------------------------------------------------
    # Simulación con ruido
    #
    # Planta:
    #
    # y'' + 1.4y' + y = u
    #
    # Control PD:
    #
    # u = kp(r-y_m) - kd dy_m/dt
    #
    # y_m = y + n
    # ---------------------------------------------------

    A = np.array([
        [0, 1],
        [-(1 + kp), -(1.4 + kd)]
    ])

    B = np.array([
        [0],
        [1]
    ])

    C = np.eye(2)

    D = np.zeros((2, 1))

    system_ss = signal.StateSpace(A, B, C, D)

    r = np.ones_like(t)

    # Entrada equivalente a la planta
    forcing = (
        kp * r
        - kp * n
        - kd * dn
    )

    tout, states, _ = signal.lsim(
        system_ss,
        U=forcing,
        T=t
    )

    y = states[:, 0]
    y_dot = states[:, 1]


    # ---------------------------------------------------
    # Señal medida y señal de control
    # ---------------------------------------------------

    y_measured = y + n

    u = (
        kp * (r - y_measured)
        - kd * (y_dot + dn)
    )


    # ---------------------------------------------------
    # Indicadores de desempeño
    # Se calculan con la respuesta sin ruido
    # ---------------------------------------------------

    y_final = kp / (1 + kp)

    ess = 1 - y_final

    y_max = np.max(y_clean)

    overshoot = max(
        0,
        (y_max - y_final) / y_final * 100
    )

    ts = settling_time(
        tout,
        y_clean,
        y_final
    )


    # ---------------------------------------------------
    # Mostrar resultados
    # ---------------------------------------------------

    with output:

        clear_output(wait=True)


        # =================================================
        # Respuesta temporal
        # =================================================

        fig, ax = plt.subplots(1, 2, figsize=(11, 4))


        ax[0].plot(
            tout,
            y,
            label='Salida y(t)'
        )

        ax[0].axhline(
            1,
            linestyle='--',
            label='Referencia'
        )

        ax[0].axhline(
            y_final,
            linestyle=':',
            label='Valor final'
        )

        ax[0].set_xlabel('Tiempo [s]')
        ax[0].set_ylabel('Salida')
        ax[0].set_title('Respuesta al escalón')
        ax[0].grid()
        ax[0].legend()


        # =================================================
        # Polos
        # =================================================

        ax[1].scatter(
            poles.real,
            poles.imag,
            marker='x',
            s=100
        )

        ax[1].axhline(0)
        ax[1].axvline(0)

        ax[1].set_xlim(-8, 1)
        ax[1].set_ylim(-3, 3)

        ax[1].set_xlabel('Parte real')
        ax[1].set_ylabel('Parte imaginaria')
        ax[1].set_title('Polos del sistema')
        ax[1].grid()

        plt.tight_layout()
        plt.show()


        # =================================================
        # Señal de control
        # =================================================

        plt.figure(figsize=(8, 3.5))

        plt.plot(
            tout,
            u
        )

        plt.xlabel('Tiempo [s]')
        plt.ylabel('u(t)')
        plt.title('Señal de control')
        plt.grid()

        plt.show()


        # =================================================
        # Resultados numéricos
        # =================================================

        print(f"kp = {kp:.2f}")
        print(f"kd = {kd:.2f}")
        print()
        print(f"Valor final y(∞)          = {y_final:.3f}")
        print(f"Error estacionario e_ss   = {ess:.3f}")
        print(f"Sobrepaso                 = {overshoot:.1f} %")

        if np.isnan(ts):
            print("T. establecimiento        > 15 s")
        else:
            print(f"T. establecimiento        = {ts:.2f} s")

        print("\nPolos:")

        for p in poles:
            print(f"   {p:.4f}")

        if noise_on:
            print("\nRuido de medición ACTIVADO")


# -------------------------------------------------------
# Actualización de controles
# -------------------------------------------------------

kd_slider.observe(
    simulate,
    names='value'
)

noise_checkbox.observe(
    simulate,
    names='value'
)

display(kd_slider)
display(noise_checkbox)
display(output)

simulate()

FloatSlider(value=0.0, continuous_update=False, description='kd:', layout=Layout(width='500px'), max=8.0, read…

Checkbox(value=False, description='Activar ruido de medición')

Output()